# Day 067 — Exercise 4: Extract Text from Image

**What you'll build:** `extract_text_from_image(img_b64, describe_fn=None)` — OCR via a vision LLM using a carefully crafted prompt.

**Why it matters:** Vision LLMs read text in context — they understand that "Qty" on a receipt means quantity. The key is the prompt: "exactly as it appears" stops the model from paraphrasing. Tomorrow (Day 68) you will compare this with pytesseract for structured documents.

In [ ]:
import io
import base64
from PIL import Image

def image_to_base64(img: Image.Image, format: str = 'PNG') -> str:
    buf = io.BytesIO()
    out = img
    if format.upper() in ('JPEG', 'JPG') and img.mode in ('RGBA', 'P'):
        out = img.convert('RGB')
    out.save(buf, format=format)
    return base64.b64encode(buf.getvalue()).decode()

import ollama

def describe_image(img_b64: str, prompt: str = 'Describe this image.',
                   describe_fn=None) -> str:
    if describe_fn is not None:
        return describe_fn(img_b64, prompt)
    resp = ollama.chat(
        model='llava',
        messages=[{'role': 'user', 'content': prompt, 'images': [img_b64]}]
    )
    return resp['message']['content']

from PIL import ImageDraw
_text_img = Image.new('RGB', (200, 60), color='white')
_draw = ImageDraw.Draw(_text_img)
_draw.text((10, 15), 'Hello World', fill='black')
_text_b64 = image_to_base64(_text_img)


## Task

Implement `extract_text_from_image(img_b64, describe_fn=None) -> str`:

- Build an OCR prompt containing `'exactly'` and `'text'`
- The prompt should handle the no-text case (instruct the model to return an empty string)
- Call `describe_image(img_b64, prompt, describe_fn=describe_fn)`
- Return the result directly

## Your Implementation

In [ ]:
def extract_text_from_image(img_b64: str, describe_fn=None) -> str:
    """Extract visible text from an image using a vision LLM.

    Uses an OCR-focused prompt that instructs the model to transcribe
    text exactly as it appears, without paraphrasing or correcting.

    Args:
        img_b64:     base64-encoded image string
        describe_fn: callable(img_b64, prompt) -> str for testing
    Returns:
        Extracted text string (may be empty if no text in image)
    """
    raise NotImplementedError


In [ ]:
def extract_text_from_image(img_b64: str, describe_fn=None) -> str:
    prompt = (
        'Extract all visible text from this image exactly as it appears. '
        'If there is no text, reply with an empty string.'
    )
    return describe_image(img_b64, prompt, describe_fn=describe_fn)


## Automated checks

In [ ]:
score, total = 0, 5
try:
    result = extract_text_from_image(_text_b64,
                                      describe_fn=lambda b, p: 'Hello World')
    assert isinstance(result, str), f"Expected str, got {type(result)}"
    score += 1; print("\u2705 returns a string")

    assert result == 'Hello World', f"Expected 'Hello World', got {result!r}"
    score += 1; print("\u2705 returns the mock's response text")

    # Prompt must contain OCR intent keywords
    captured = {}
    def _mock_capture(b, p):
        captured['prompt'] = p
        return ''
    extract_text_from_image(_text_b64, describe_fn=_mock_capture)
    prompt_lower = captured.get('prompt', '').lower()
    assert 'text' in prompt_lower or 'extract' in prompt_lower, (
        f"Prompt should mention 'text' or 'extract': {captured.get('prompt')!r}")
    score += 1; print("\u2705 prompt contains OCR intent keyword")

    # Prompt contains 'exactly' to enforce literal transcription
    assert 'exactly' in prompt_lower or 'literal' in prompt_lower or 'appears' in prompt_lower, (
        f"Prompt should contain 'exactly' or 'appears': {captured.get('prompt')!r}")
    score += 1; print("\u2705 prompt enforces exact transcription")

    # Empty string returned for image with no text
    empty = extract_text_from_image(_text_b64,
                                     describe_fn=lambda b, p: '')
    assert isinstance(empty, str), "Should return empty string for no-text image"
    score += 1; print("\u2705 handles empty-text response gracefully")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def extract_text_from_image(img_b64: str, describe_fn=None) -> str:
    prompt = (
        'Extract all visible text from this image exactly as it appears. '
        'If there is no text, reply with an empty string.'
    )
    return describe_image(img_b64, prompt, describe_fn=describe_fn)
```

**Why `'exactly as it appears'`?** Without this constraint, vision LLMs often paraphrase — they might correct a typo, expand an abbreviation, or reformat a date. For OCR you want the literal characters. Telling the model to return an empty string for no-text images prevents responses like 'I do not see any text in this image.' which would require post-processing to detect.

</details>